# MGSRL + MSTRL Encoder Pipeline

This notebook now uses the real feature tensor from the GHI preprocessing pipeline together with the graph artifacts produced by the precompute_graphs workflow instead of synthetic data placeholders.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

if importlib.util.find_spec('torch_geometric') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch-geometric'])

root = Path.cwd().resolve()
candidate_roots = [root]
if not (root / 'Src').exists() and (root.parent / 'Src').exists():
    candidate_roots.append(root.parent)

for candidate in candidate_roots:
    if (candidate / 'Src').exists():
        sys.path.insert(0, str(candidate))
        break

from Src.spatial_gatv2_encoder import SpatialGATv2EncoderBatched
from Src.fusion_module import SpatialFusionModule
from Src.temporal_conv_module import TemporalConvModule
from Src.data_splits import load_windowed_drive_folds
from Src.temporal_graph_state import TemporalGraphStateAggregator, LatentGraphGenerator


def resolve_real_data_dir(base_dir=None):
    candidates = []
    if base_dir is not None:
        candidates.append(Path(base_dir))

    candidates.extend(
        [
            root / 'filtered_data',
            root.parent / 'filtered_data',
            root / 'data',
            root.parent / 'data',
            Path('/content/gdrive/MyDrive/filtered_data'),
        ]
    )

    for candidate in candidates:
        if candidate.exists() and (candidate / 'filtered_tensor.npz').exists() and (candidate / 'precomputed_graphs').exists():
            return str(candidate)

    raise FileNotFoundError(
        'Could not locate the real data directory. Point base_dir to a folder that contains filtered_tensor.npz and precomputed_graphs.'
    )


In [ ]:
def build_mgsrc_mstrl_pipeline(
    base_dir: str | None = None,
    batch_size: int = 2,
    time_steps: int = 6,
    input_dim: int = 8,
    spatial_hidden_dim: int = 16,
    temporal_hidden_dim: int = 32,
    W: int = 6,
    H: int = 1,
    stride: int = 1,
):
    torch.manual_seed(42)

    real_data_dir = resolve_real_data_dir(base_dir)
    folds = load_windowed_drive_folds(
        real_data_dir,
        W=W,
        H=H,
        stride=stride,
        train_duration=None,
        val_duration=0,
        test_duration=365,
        fold_count=1,
        input_mask_key='mask_forecast',
        output_mask_key='mask_forecast',
    )
    fold = folds[0]
    train_split = fold.train

    x_seq = torch.from_numpy(train_split.x[:batch_size]).float()
    if x_seq.ndim == 4:
        x_seq = x_seq[:, :time_steps]
    elif x_seq.ndim == 3:
        x_seq = x_seq[:, :time_steps]
        x_seq = x_seq.unsqueeze(1)
    else:
        raise ValueError(f'Unexpected input tensor shape: {x_seq.shape}')

    graph_names = sorted(train_split.historical_graphs.keys())
    edge_indices = []
    edge_weights = []
    adjacency_inputs = []
    for graph_name in graph_names:
        graph = train_split.historical_graphs[graph_name]
        graph_batch = graph[:batch_size]
        if graph_batch.ndim == 4:
            adj = graph_batch[0, 0]
        elif graph_batch.ndim == 3:
            adj = graph_batch[0]
        else:
            adj = graph_batch

        adj = torch.from_numpy(np.asarray(adj)).float()
        adjacency_inputs.append(adj)
        num_nodes = adj.shape[0]
        edge_index = torch.tensor(
            [[i, j] for i in range(num_nodes) for j in range(num_nodes) if i != j],
            dtype=torch.long,
        ).t()
        edge_weight = adj[edge_index[0], edge_index[1]]
        edge_indices.append(edge_index)
        edge_weights.append(edge_weight)

    spatial_encoder = SpatialGATv2EncoderBatched(
        in_features=x_seq.shape[-1],
        out_features=spatial_hidden_dim,
        n_graphs=len(graph_names),
        heads=4,
        dropout=0.1,
    )
    fusion = SpatialFusionModule(
        embedding_dim=spatial_hidden_dim,
        n_graphs=len(graph_names),
        fusion_method='attention',
        dropout=0.1,
    )
    temporal_encoder = TemporalConvModule(
        node_feature_dim=spatial_hidden_dim,
        hidden_dim=temporal_hidden_dim,
        dilation_list=[1, 2, 4],
        kernel_size=3,
        dropout=0.1,
    )

    fused_sequence = []
    for t in range(min(time_steps, x_seq.shape[1])):
        x_t = x_seq[:, t]
        graph_embeddings = spatial_encoder(x_t, edge_indices, edge_weights)
        fused_t = fusion(graph_embeddings)
        fused_sequence.append(fused_t)

    spatial_sequence = torch.stack(fused_sequence, dim=1)
    Z = temporal_encoder(spatial_sequence)

    state_aggregator = TemporalGraphStateAggregator(channels=temporal_hidden_dim)
    S = state_aggregator(Z)

    graph_generator = LatentGraphGenerator(state_dim=S.shape[-1], hidden_dim=32)
    future_graphs = []
    for edge_index, historical_adj in zip(edge_indices, adjacency_inputs):
        historical_adj_batch = historical_adj.unsqueeze(0).repeat(batch_size, 1, 1)
        future_edge_weights = graph_generator(historical_adj_batch, S, edge_index)
        future_graphs.append(future_edge_weights)

    return {
        'x_seq': x_seq,
        'spatial_sequence': spatial_sequence,
        'Z': Z,
        'S': S,
        'future_graphs': future_graphs,
        'spatial_encoder': spatial_encoder,
        'fusion': fusion,
        'temporal_encoder': temporal_encoder,
        'state_aggregator': state_aggregator,
        'graph_generator': graph_generator,
        'fold': fold,
        'graph_names': graph_names,
        'data_dir': real_data_dir,
    }


In [ ]:
pipeline = build_mgsrc_mstrl_pipeline(base_dir=None)
Z = pipeline['Z']
S = pipeline['S']
print('Latent matrix Z shape:', tuple(Z.shape))
print('Compact graph state S shape:', tuple(S.shape))
print('First time-step summary:', Z[0, 0, :5, :5])
print('Compact state summary:', S[0, :5, :5])
print('Loaded data dir:', pipeline['data_dir'])
print('Graphs used:', pipeline['graph_names'])

for idx, future_graph in enumerate(pipeline['future_graphs']):
    print(f'Future edge weights for graph {idx}:', tuple(future_graph.shape))

# Optional: reconstruct a graph from the learned latent matrix.
z_summary = Z.mean(dim=1)
sim_matrix = torch.cdist(z_summary, z_summary, p=2)
reconstructed_adj = (sim_matrix < 1.0).float()
print('Reconstructed adjacency shape:', tuple(reconstructed_adj.shape))
